# One Small Change, a Completely Different Hash
A good hash function changes its whole output when the input changes even slightly. This is called the **avalanche effect**.

## Step 1: Change one character
Each message below differs from "hello" by only one character.

In [1]:
import hashlib

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

for message in ["hello", "Hello", "hello.", "hellp"]:
    print(f"{message!r:9} {sha256_text(message)}")

'hello'   2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
'Hello'   185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969
'hello.'  1589999b0ca6ef8814283026a9f166d51c70a910671c3d44049755f07f2eb910
'hellp'   fdd7585e08c4e2afd71dcabdb4636c89d557a3f42db9e2040c8bbd1708aa4ce7


## Step 2: Count how many bits changed
"hello" and "Hello" differ by only **1 bit** in the input. We turn both hashes into bits and count how many positions are different.

In [2]:
def to_bits(data: bytes) -> str:
    # each byte becomes 8 bits, using the 08b code from 01-bits-bytes-hex
    return "".join(format(byte, "08b") for byte in data)

def count_diff(a: str, b: str) -> int:
    # strict=True raises an error if the two lengths are different
    return sum(1 for x, y in zip(a, b, strict=True) if x != y)

# The inputs: h and H
print("h:", format(ord("h"), "08b"))
print("H:", format(ord("H"), "08b"))

# The outputs
a = to_bits(hashlib.sha256(b"hello").digest())
b = to_bits(hashlib.sha256(b"Hello").digest())

changed = count_diff(a, b)
print(f"{changed} of {len(a)} bits changed")

h: 01101000
H: 01001000
125 of 256 bits changed


## Step 3: Test many messages
One result could be luck. Here we compare the hash of "hello" with the hashes of 1,000 similar messages ("hello0", "hello1", ...) and look at the average number of bits that change.

In [3]:
base_bits = to_bits(hashlib.sha256(b"hello").digest())

results = []
for i in range(1000):
    other_bits = to_bits(hashlib.sha256(f"hello{i}".encode("utf-8")).digest())
    changed = sum(1 for x, y in zip(base_bits, other_bits) if x != y)
    results.append(changed)

print("Average bits changed:", sum(results) / len(results))
print("Fewest:", min(results))
print("Most:", max(results))

Average bits changed: 127.895
Fewest: 106
Most: 160


## Step 4: Encoding vs hashing
Encoding is a direct translation: each letter becomes its own byte, one after the other. So changing one letter only changes that letter's bits.

Hashing is different. Every output bit depends on all the input bits, so changing one input bit changes about half of the output.

Here we compare "hello" and "Hello" both ways.

In [4]:
# Encoding: a direct translation, so 1 bit in = 1 bit out
encoded_a = to_bits(b"hello")
encoded_b = to_bits(b"Hello")
print("Encoded:", count_diff(encoded_a, encoded_b), "of", len(encoded_a), "bits differ")

# Hashing: everything is mixed, so 1 bit in = about half the bits out
hashed_a = to_bits(hashlib.sha256(b"hello").digest())
hashed_b = to_bits(hashlib.sha256(b"Hello").digest())
print("Hashed: ", count_diff(hashed_a, hashed_b), "of", len(hashed_a), "bits differ")

Encoded: 1 of 40 bits differ
Hashed:  125 of 256 bits differ


## Step 5: Behind the scenes: how one bit spreads
**This is a toy example, not real SHA-256, and it is not secure.** It only shows the idea.

SHA-256 repeats a few simple operations many times:
- **Rotate:** slide all the bits left. Bits that fall off the left come back on the right.
- **XOR (`^`):** combine two bit patterns. The result is 1 where the bits differ and 0 where they match.
- **Add:** normal addition. Carries can spread a change into the next bits, like 199 + 1 changing three digits.

Each operation turns one changed bit into a few. The toy below does one round of three steps on 16 bits ("hi" vs "Hi") and shows the difference growing.

Notes on the code:
- `0b0110...` types a number in binary, and `0x9E37` types a number in hex.
- `<<` and `>>` shift bits left and right. Together they make the rotate.
- `& 0xFFFF` keeps only the last 16 bits, so the number doesn't grow bigger.
- `bin(a ^ b).count("1")` counts differing bits directly on the numbers, without making strings.

In [6]:
def rotate_left(x: int, r: int) -> int:
    # slide bits r places left; bits that fall off the left come back on the right
    return ((x << r) | (x >> (16 - r))) & 0xFFFF

def show(label: str, a: int, b: int) -> None:
    diff = bin(a ^ b).count("1")   # XOR marks differing bits with 1, then we count the 1s
    print(f"{label:22} {a:016b}")
    print(f"{'':22} {b:016b}   {diff} bit(s) differ\n")

a = 0b0110100001101001   # "hi"
b = 0b0100100001101001   # "Hi" (1 bit different)
show("Start", a, b)

a, b = (a + 0x9E37) & 0xFFFF, (b + 0x9E37) & 0xFFFF
show("1. Add a constant", a, b)

a, b = a ^ rotate_left(a, 5), b ^ rotate_left(b, 5)
show("2. XOR with rotated", a, b)

a, b = (a + rotate_left(a, 11)) & 0xFFFF, (b + rotate_left(b, 11)) & 0xFFFF
show("3. Add rotated", a, b)

Start                  0110100001101001
                       0100100001101001   1 bit(s) differ

1. Add a constant      0000011010100000
                       1110011010100000   3 bit(s) differ

2. XOR with rotated    1101001010100000
                       0011001010111100   6 bit(s) differ

3. Add rotated         1101100100110101
                       0001010001010001   8 bit(s) differ



In [40]:
print(sum(v for v in range(1, 6)))
print(int(5))
print(int("10110110", 2))

15
5
182


In [12]:
print(int("101", 2))   # 5    read as binary
print(int("ff", 16))   # 255  read as hex
print(int("10"))       # 10   no base given, so normal base 10
print(int("10", 2))    # 2    the same "10", read as binary

5
255
10
2


## Summary
- Changing even one bit of the input changes about half of the 256 output bits (125 in my test of "hello" vs "Hello").
- Over many messages, the average is very close to 128, which is half.
- Half is the ideal: the two hashes look completely unrelated, so the hash gives away nothing about how similar two messages are.
- Encoding is a direct translation (1 bit in, 1 bit out). Hashing mixes everything (1 bit in, about half the bits out).
- The mixing comes from repeating simple steps (rotate, XOR, add) many times.
- Comparing bits here was only a way to measure the difference. To verify a hash, we just check the whole thing matches exactly.